# A marcenaria Gralha Azul

**Fonte:** Arce, J. E. (2025). *Programação Linear para Fins Florestais*. Apostila da disciplina AS062, UFPR. Exemplo 3, p. 19–22.

**Neste notebook:** minimização com restrições do tipo "≥": gastar o mínimo de energia garantindo emprego e faturamento. Mesmo problema da planilha `02_marcenaria.xlsx`.

**Sua tarefa:** completar as restrições do modelo. Onde aparecer `____`, escreva o número certo (por exemplo, `10`) ou o nome do dado (por exemplo, `mao_obra_pinus`); os dois funcionam. A primeira restrição vem pronta, como exemplo.

**Se aparecer `NameError: name '____' is not defined`**, ainda falta preencher alguma lacuna.

**Como rodar:** menu **Ambiente de execução → Executar tudo**. Os passos são os mesmos da planilha:
**estimar → modelar → resolver → comparar**.

## 0. Preparar o ambiente

Instala o Pyomo (a linguagem do modelo) e o HiGHS (o *solver*). O Colab apaga tudo ao fechar a aba,
então esta célula roda sempre que o notebook é aberto.

In [ ]:
%pip install -q pyomo highspy

Abaixo ficam três ferramentas para resolver e ler o resultado em tabelas.
**Você não precisa entender este código:** basta rodar.

In [ ]:
import pyomo.environ as pyo
import pandas as pd


def br(numero, casas=2):
    """Escreve um número no formato brasileiro: 1.234,56."""
    texto = f"{numero:,.{casas}f}"
    return texto.replace(",", "X").replace(".", ",").replace("X", ".")


def resolver(modelo):
    """Chama o HiGHS, mostra o status e devolve o valor do objetivo."""
    resultado = pyo.SolverFactory("appsi_highs").solve(modelo)
    print("Status:", resultado.solver.termination_condition)
    objetivo = next(modelo.component_objects(pyo.Objective, active=True))
    print("Objetivo:", br(pyo.value(objetivo)))
    return pyo.value(objetivo)


def ler_variaveis(modelo):
    """Tabela com o valor de cada variável de decisão."""
    linhas = []
    for variavel in modelo.component_objects(pyo.Var, active=True):
        for indice in variavel:
            nome = variavel.name if indice is None else f"{variavel.name}[{indice}]"
            linhas.append({"variável": nome, "valor": round(pyo.value(variavel[indice]), 2)})
    return pd.DataFrame(linhas)


def ler_restricoes(modelo):
    """Tabela com o uso, o limite e a folga de cada restrição (como a aba 4_comparar)."""
    linhas = []
    for restricao in modelo.component_objects(pyo.Constraint, active=True):
        for indice in restricao:
            r = restricao[indice]
            usado = pyo.value(r.body)
            if r.equality:
                sinal, limite, folga = "=", pyo.value(r.upper), usado - pyo.value(r.upper)
            elif r.has_ub():
                sinal, limite, folga = "<=", pyo.value(r.upper), pyo.value(r.upper) - usado
            else:
                sinal, limite, folga = ">=", pyo.value(r.lower), usado - pyo.value(r.lower)
            nome = restricao.name if indice is None else f"{restricao.name}[{indice}]"
            linhas.append({
                "restrição": nome,
                "usado": round(usado, 2),
                "sinal": sinal,
                "limite": round(limite, 2),
                "folga": round(folga, 2),
                "limitou?": "Sim" if abs(folga) < 1e-6 else "Não",
            })
    return pd.DataFrame(linhas)

## Enunciado

A marcenaria Gralha Azul fabrica conjuntos de móveis (1 mesa e 4 cadeiras) e kits de esquadrias
(1 porta e 3 janelas). Uma estiagem levou ao racionamento de energia: o dono quer gastar o mínimo de
energia elétrica por dia. Com a crise, decidiu manter os 10 colaboradores trabalhando (10 diárias por dia)
e precisa faturar pelo menos R$ 4.000 por dia. Cada linha de produção faz no máximo 4 unidades por dia.

| Item | Conjunto de móveis | Kit de esquadrias | Exigência |
|---|---:|---:|---|
| Preço de venda (R$/unidade) | 1.200 | 800 | pelo menos R$ 4.000/dia |
| Funcionários (diárias/unidade) | 2 | 3 | pelo menos 10 diárias/dia |
| Energia (kWh/unidade) | 8 | 10 | gastar o mínimo |
| Capacidade (unidades/dia) | 4 | 4 | |

**A pergunta:** quantos conjuntos e quantos kits fazer por dia para gastar o mínimo de energia?

## Dados

In [ ]:
# Energia (kWh/unidade)
energia_moveis = 8
energia_esquadrias = 10

# Diárias de funcionários por unidade e mínimo a garantir (diárias/dia)
diarias_moveis = 2
diarias_esquadrias = 3
diarias_minimo = 10

# Preço de venda (R$/unidade) e faturamento mínimo (R$/dia)
preco_moveis = 1200
preco_esquadrias = 800
faturamento_minimo = 4000

# Capacidade de cada linha (unidades/dia)
capacidade_moveis = 4
capacidade_esquadrias = 4

## 1. Estimar antes de modelar

Qual o menor consumo de energia só para manter os 10 colaboradores trabalhando?
Basta usar o produto que gasta menos energia por diária. Como a conta ignora o faturamento, é um **piso**.

In [ ]:
# Energia por diária de cada produto (kWh/diária)
kwh_por_diaria_moveis = energia_moveis / diarias_moveis
kwh_por_diaria_esquadrias = energia_esquadrias / diarias_esquadrias

# Piso: 10 diárias com o produto que gasta menos por diária
piso = diarias_minimo * min(kwh_por_diaria_moveis, kwh_por_diaria_esquadrias)
print("Piso da energia:", br(piso, 1), "kWh/dia")

## 2. Modelar

$$
\begin{aligned}
\min \quad & z = 8\,x_1 + 10\,x_2 && \text{(energia, kWh/dia)}\\
\text{sujeito a} \quad & 2\,x_1 + 3\,x_2 \ge 10 && \text{(emprego, diárias/dia)}\\
& 1.200\,x_1 + 800\,x_2 \ge 4.000 && \text{(faturamento, R\$/dia)}\\
& x_1 \le 4 && \text{(capacidade de móveis)}\\
& x_2 \le 4 && \text{(capacidade de esquadrias)}\\
& x_1,\ x_2 \ge 0
\end{aligned}
$$

$x_1$ = conjuntos de móveis por dia; $x_2$ = kits de esquadrias por dia.
A planilha e o livro escrevem o faturamento em notas de R$ 100 ($12\,x_1 + 8\,x_2 \ge 40$);
aqui usamos reais. A solução é a mesma.

**Variáveis de decisão:** $x_1, x_2 \ge 0$ (unidades/dia).

In [ ]:
modelo = pyo.ConcreteModel("marcenaria")

# Unidades de cada produto fabricadas por dia (não negativas)
modelo.x_moveis = pyo.Var(domain=pyo.NonNegativeReals)
modelo.x_esquadrias = pyo.Var(domain=pyo.NonNegativeReals)

**Objetivo:** $\min\ z = 8\,x_1 + 10\,x_2$. Atenção: aqui é **minimizar**.

In [ ]:
@modelo.Objective(sense=pyo.minimize)
def energia(m):
    return energia_moveis * m.x_moveis + energia_esquadrias * m.x_esquadrias

**Emprego:** $2\,x_1 + 3\,x_2 \ge 10$. É o "≥" que obriga a produzir alguma coisa.

In [ ]:
@modelo.Constraint()
def emprego(m):
    return diarias_moveis * m.x_moveis + diarias_esquadrias * m.x_esquadrias >= diarias_minimo

**Faturamento:** $1.200\,x_1 + 800\,x_2 \ge 4.000$

In [ ]:
@modelo.Constraint()
def faturamento(m):
    return ____ * m.x_moveis + ____ * m.x_esquadrias >= ____

**Capacidade das linhas:** $x_1 \le 4$ e $x_2 \le 4$

In [ ]:
@modelo.Constraint()
def capacidade_moveis_un(m):
    return m.x_moveis <= ____

@modelo.Constraint()
def capacidade_esquadrias_un(m):
    return m.x_esquadrias <= ____

## 3. Resolver

In [ ]:
otimo = resolver(modelo)

In [ ]:
ler_variaveis(modelo)

## 4. Comparar

In [ ]:
print("Piso (estimativa):", br(piso, 1), "kWh/dia")
print("Ótimo (HiGHS):    ", br(otimo, 1), "kWh/dia")
print("Diferença:        ", br(otimo - piso, 1), "kWh/dia")

In [ ]:
ler_restricoes(modelo)

## 5. Perguntas

Dê dois cliques nesta célula para escrever as suas respostas.

**1. Por que este modelo precisa de restrições "≥"?**

> Sua resposta:

**2. Quais restrições limitaram a solução?**

> Sua resposta:

**3. O ótimo ficou acima do piso da etapa 1? Por quê?**

> Sua resposta:

## 6. Mude e rode de novo

Troque um número na célula de **dados** (por exemplo, um limite ou um preço) e rode tudo de novo:
**Ambiente de execução → Executar tudo**. O que mudou na solução? E nas folgas?